# 04 Label Sentiment

This notebook creates and manually labels the fixed sentiment-evaluation subset.


## Fixed sample

The submitted specification requires a fixed manually labelled subset but does not prescribe its size. For this MVP, the notebook selects **300 headlines**: 30 randomly sampled headlines from each of the ten companies. The fixed random seed `2025` makes the selection reproducible and company balancing prevents firms with more GDELT coverage from dominating the labelled subset.

Labels are assigned to the target company using only information explicitly stated in the headline. This follows the entity-aware, investor-perspective approach described by [Sinha et al. (2022)](https://doi.org/10.1002/asi.24634).


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)


In [2]:
# Support running the notebook from either the project root or the pipeline directory.
working_directory = Path.cwd()
project_root = (
    working_directory.parent
    if working_directory.name == "pipeline"
    else working_directory
)

aligned_path = project_root / "data" / "processed" / "headlines_aligned_prices.csv"
gold_labels_path = project_root / "data" / "processed" / "gold_labels.csv"

print("Project root:", project_root)
print("Aligned data exists:", aligned_path.exists())


Project root: c:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks
Aligned data exists: True


In [3]:
aligned_df = pd.read_csv(aligned_path)

required_columns = {
    "headline_id",
    "published_at_utc",
    "published_at_london",
    "ticker",
    "company_name",
    "headline_text",
}

assert required_columns.issubset(aligned_df.columns)
assert aligned_df["headline_id"].is_unique
assert not aligned_df[list(required_columns)].isna().any().any()
assert aligned_df["ticker"].nunique() == 10

print(f"Available aligned headlines: {len(aligned_df):,}")


Available aligned headlines: 9,001


In [4]:
SAMPLE_PER_COMPANY = 30
RANDOM_SEED = 2025

assert aligned_df.groupby("ticker").size().min() >= SAMPLE_PER_COMPANY

sample_df = (
    aligned_df.groupby("ticker", group_keys=False)
    .sample(n=SAMPLE_PER_COMPANY, random_state=RANDOM_SEED)
    .sort_values(["ticker", "published_at_utc", "headline_text"])
    .reset_index(drop=True)
)

assert len(sample_df) == 300
assert sample_df.groupby("ticker").size().eq(SAMPLE_PER_COMPANY).all()
assert sample_df["headline_id"].is_unique

display(sample_df.groupby("ticker").size().rename("sampled_headlines"))


ticker
AZN.L     30
GSK.L     30
HLMA.L    30
HSBA.L    30
ITRK.L    30
NG.L      30
REL.L     30
RIO.L     30
SN.L      30
VOD.L     30
Name: sampled_headlines, dtype: int64

## Labelling guide

Judge the likely financial effect on the **named target company**, using only the headline. Do not use the later share price, article body, or outside knowledge.

- **positive**: the headline clearly indicates a favourable development, such as improved results, an approval, a contract win, an upgrade, or another likely benefit.
- **negative**: the headline clearly indicates an unfavourable development, such as weaker results, a warning, a rejection, litigation, disruption, a downgrade, or another likely harm.
- **neutral**: the headline is factual, mixed, unclear, or does not indicate a clear positive or negative effect on the company.



In [5]:
display_columns = [
    "headline_id",
    "ticker",
    "company_name",
    "published_at_london",
    "headline_text",
]

labelling_df = sample_df[display_columns].copy()

if gold_labels_path.exists():
    existing_labels = pd.read_csv(gold_labels_path, keep_default_na=False)
    expected_gold_columns = [
        "headline_id", "label_gold", "annotator_pass"
    ]
    assert existing_labels.columns.tolist() == expected_gold_columns
    assert existing_labels["headline_id"].is_unique
    assert set(existing_labels["headline_id"]).issubset(
        set(labelling_df["headline_id"])
    )
    labelling_df = labelling_df.merge(
        existing_labels, on="headline_id", how="left"
    )
    labelling_df["label_gold"] = labelling_df["label_gold"].fillna("")
    labelling_df["annotator_pass"] = (
        labelling_df["annotator_pass"].fillna(1).astype(int)
    )
else:
    labelling_df["label_gold"] = ""
    labelling_df["annotator_pass"] = 1

print("Existing label progress loaded.")


Existing label progress loaded.


## Initial labels

The cells below assign a headline-only label to every sampled headline. They are separated by ticker so the decisions are easy to inspect. Rerunning these cells fills only blank labels and does not overwrite labels already saved in `gold_labels.csv`.


In [ ]:
def assign_labels(indices, label):
    """Assign a label to blank rows without overwriting saved work."""

    rows_are_selected = labelling_df.index.isin(indices)
    labels_are_blank = labelling_df["label_gold"] == ""
    rows_to_label = rows_are_selected & labels_are_blank

    labelling_df.loc[rows_to_label, "label_gold"] = label
    labelling_df.loc[rows_to_label, "annotator_pass"] = 1


In [ ]:
# AstraZeneca (AZN.L): rows 0-29
assign_labels(
    [0, 1, 2, 6, 9, 10, 12, 16, 17, 18, 19, 23, 25, 27, 28, 29],
    "positive",
)
assign_labels([3, 4, 7, 8, 11, 14, 15, 21, 24], "negative")
assign_labels([5, 13, 20, 22, 26], "neutral")


In [ ]:
# GSK (GSK.L): rows 30-59
assign_labels(
    [31, 33, 35, 36, 37, 38, 41, 43, 44, 45, 46, 47, 50, 52, 56, 57],
    "positive",
)
assign_labels([34, 39, 40, 42, 49, 55, 58], "negative")
assign_labels([30, 32, 48, 51, 53, 54, 59], "neutral")


In [ ]:
# Halma (HLMA.L): rows 60-89
assign_labels(
    [64, 65, 66, 67, 68, 69, 70, 72, 73, 74, 75, 76, 77, 79, 80, 81, 82, 83, 84, 88],
    "positive",
)
assign_labels([62, 78, 85], "negative")
assign_labels([60, 61, 63, 71, 86, 87, 89], "neutral")


In [ ]:
# HSBC (HSBA.L): rows 90-119
assign_labels([90, 92, 93, 96, 97, 100, 103, 111, 115], "positive")
assign_labels([94, 95, 99, 102, 109], "negative")
assign_labels(
    [91, 98, 101, 104, 105, 106, 107, 108, 110, 112, 113, 114, 116, 117, 118, 119],
    "neutral",
)


In [ ]:
# Intertek (ITRK.L): rows 120-149
assign_labels(
    [120, 121, 122, 123, 124, 125, 126, 130, 132, 133, 135, 136, 137, 141, 142, 143, 145, 146, 148],
    "positive",
)
assign_labels([128, 129, 134, 140, 144], "negative")
assign_labels([127, 131, 138, 139, 147, 149], "neutral")


In [ ]:
# National Grid (NG.L): rows 150-179
assign_labels([157, 162, 163, 165, 170, 171, 177, 179], "positive")
assign_labels([150, 152, 153, 158, 159, 164, 166, 167, 168], "negative")
assign_labels(
    [151, 154, 155, 156, 160, 161, 169, 172, 173, 174, 175, 176, 178],
    "neutral",
)


In [ ]:
# RELX (REL.L): rows 180-209
assign_labels(
    [180, 181, 182, 183, 184, 187, 189, 190, 191, 192, 194, 195, 196, 201, 202, 203, 204, 206, 207, 208],
    "positive",
)
assign_labels([185, 186, 193, 197, 209], "negative")
assign_labels([188, 198, 199, 200, 205], "neutral")


In [ ]:
# Rio Tinto (RIO.L): rows 210-239
assign_labels(
    [215, 216, 219, 220, 221, 225, 227, 230, 232, 233, 237, 238, 239],
    "positive",
)
assign_labels([210, 217, 218, 223, 224, 226, 228, 236], "negative")
assign_labels([211, 212, 213, 214, 222, 229, 231, 234, 235], "neutral")


In [ ]:
# Smith & Nephew (SN.L): rows 240-269
assign_labels(
    [240, 241, 242, 243, 244, 246, 247, 248, 253, 259, 260, 261, 262, 264, 265, 266, 267, 268],
    "positive",
)
assign_labels([249, 252, 254, 269], "negative")
assign_labels([245, 250, 251, 255, 256, 257, 258, 263], "neutral")


In [ ]:
# Vodafone (VOD.L): rows 270-299
assign_labels(
    [271, 273, 275, 277, 283, 285, 286, 287, 288, 289, 294, 296, 297],
    "positive",
)
assign_labels([272, 274, 276, 280, 281, 282, 291, 292, 295, 298], "negative")
assign_labels([270, 278, 279, 284, 290, 293, 299], "neutral")


## Inspect one company at a time

Change `TICKER_TO_INSPECT` and run the display cell. If a label is incorrect, update it using its displayed row number and save the notebook progress again.


In [ ]:
TICKER_TO_INSPECT = "AZN.L"

display(
    labelling_df.loc[
        labelling_df["ticker"] == TICKER_TO_INSPECT,
        [
            "ticker",
            "company_name",
            "headline_text",
            "label_gold",
            "annotator_pass",
        ],
    ]
)


In [ ]:
# Correct labels by using the row numbers displayed above.
# labelling_df.loc[[0, 1], "label_gold"] = "neutral"


In [8]:
valid_labels = {"positive", "neutral", "negative"}

entered_labels = set(labelling_df["label_gold"])
invalid_labels = entered_labels - valid_labels - {""}

assert not invalid_labels, f"Invalid labels: {sorted(invalid_labels)}"
assert labelling_df["annotator_pass"].isin([1, 2]).all()

unlabelled_count = int((labelling_df["label_gold"] == "").sum())

print(f"Labelled: {len(labelling_df) - unlabelled_count:,} / {len(labelling_df):,}")
display(labelling_df["label_gold"].value_counts(dropna=False))


Labelled: 0 / 300
Still flagged for review: 0


label_gold
    300
Name: count, dtype: int64

In [9]:
# Save progress using exactly the gold-label schema from the specification.
gold_columns = [
    "headline_id", "label_gold", "annotator_pass"
]
gold_labels_df = labelling_df[gold_columns].copy()

gold_labels_path.parent.mkdir(parents=True, exist_ok=True)
gold_labels_df.to_csv(gold_labels_path, index=False)

print(f"Saved label progress to {gold_labels_path}")


Saved label progress to c:\Users\PC\Desktop\Masters\09 Capstone Project - Dissertation\Week 40\DK-CoT_UK_Stocks\data\processed\gold_labels.csv


In [10]:
if unlabelled_count == 0:
    print("Labelling complete: gold_labels.csv is ready for model evaluation.")
else:
    print("Labelling is not complete. Finish the blank labels.")


Labelling is not complete. Finish blank labels and resolve QA review flags.
